# 08 - MobileNetV2 Model Training

**Goal:** Build, train, and evaluate a MobileNetV2-based transfer learning model for multi-crop disease detection.

| Phase | Step |
|-------|------|
| 9.1 | Load Dataset |
| 9.2 | Build Data Pipeline |
| 9.3 | Load MobileNetV2 |
| 9.4 | Build Model |
| 9.5 | Compile Model |
| 9.6 | Train Model |
| 9.7 | Save Model |
| 9.8 | Evaluate Model |

| Info | Value |
|------|-------|
| Dataset | PlantVillage (Split) |
| Model | MobileNetV2 (Transfer Learning) |
| Classes | 38 |
| Input Size | 224×224 |

## Phase 9.1 — Load Dataset

In [ ]:
# Cell 1 - Import Libraries
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

In [ ]:
# Cell 2 - Dataset Paths
train_dir = "../dataset/train"
val_dir   = "../dataset/val"
test_dir  = "../dataset/test"

print("Train path      :", train_dir)
print("Validation path :", val_dir)
print("Test path       :", test_dir)

In [ ]:
# Cell 3 - Image Settings
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

print("Image Size :", IMAGE_SIZE)
print("Batch Size :", BATCH_SIZE)

In [ ]:
# Cell 4 - Load Training Dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True
)

In [ ]:
# Cell 5 - Load Validation Dataset
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

In [ ]:
# Cell 6 - Load Test Dataset
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

In [ ]:
# Cell 7 - Check Classes
class_names = train_dataset.class_names

print("Total Classes:", len(class_names))
print()
for i, name in enumerate(class_names):
    print(f"  {i+1:>2}. {name}")

In [ ]:
# Cell 8 - Show Dataset Information
print("=" * 45)
print("Dataset Summary")
print("=" * 45)
print(f"Training Batches   : {len(train_dataset)}")
print(f"Validation Batches : {len(validation_dataset)}")
print(f"Test Batches       : {len(test_dataset)}")
print(f"Batch Size         : {BATCH_SIZE}")
print(f"Image Size         : {IMAGE_SIZE}")
print(f"Total Classes      : {len(class_names)}")
print("=" * 45)

In [ ]:
# Cell 9 - Display Sample Images from Training Set
plt.figure(figsize=(12, 12))

for images, labels in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        class_label = class_names[tf.argmax(labels[i])]
        crop, condition = class_label.split("___") if "___" in class_label else (class_label, "")
        plt.title(f"{crop}\n{condition}", fontsize=8)
        plt.axis("off")

plt.suptitle("Training Dataset — Sample Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_training_images.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Phase 9.1 Complete — Dataset loaded successfully!")
print("✅ Ready for Phase 9.2 — Build Data Pipeline")

## Phase 9.2 — Build Data Pipeline

In [13]:
# Cell 10 - Cache Dataset
train_dataset = train_dataset.cache()
validation_dataset = validation_dataset.cache()
test_dataset = test_dataset.cache()

print("Caching enabled on datasets")

Caching enabled on datasets


In [14]:
# Cell 11 - Enable Prefetch
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

print("Prefetch enabled on datasets using AUTOTUNE")

Prefetch enabled on datasets using AUTOTUNE


In [15]:
# Cell 12 - Verify Pipeline
print(train_dataset)
print(validation_dataset)
print(test_dataset)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 38), dtype=tf.float32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 38), dtype=tf.float32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 38), dtype=tf.float32, name=None))>


## Phase 9.3 — Load MobileNetV2 (Transfer Learning)

**Goal:** Load the pre-trained MobileNetV2 model with weights trained on the ImageNet dataset, freeze its weights, and verify its structure.

In [16]:
# Cell 13 - Import Keras Application Modules
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers
from tensorflow.keras import models

In [17]:
# Cell 14 - Load Pre-trained MobileNetV2 Base Model
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 7s 1us/step


In [18]:
# Cell 15 - Freeze Base Model Weights
base_model.trainable = False

In [19]:
# Cell 16 - Verify Base Model Structure and Trainable Parameters
base_model.summary()

Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

## Phase 9.4 — Build Classification Head

**Goal:** Build a custom classification head on top of the frozen MobileNetV2 base model using Global Average Pooling, Dropout, and Dense layers.

In [20]:
# Cell 17 - Build classification head on top of base model
model = models.Sequential([
    # Pretrained MobileNetV2
    base_model,

    # Convert feature maps to a vector
    layers.GlobalAveragePooling2D(),

    # Prevent overfitting
    layers.Dropout(0.3),

    # Hidden layer
    layers.Dense(128, activation="relu"),

    # Extra dropout
    layers.Dropout(0.2),

    # Output layer (38 disease classes)
    layers.Dense(38, activation="softmax")
])

In [21]:
# Cell 18 - Print Complete Model Summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,854 (9.26 MB)

 Trainable params: 168,870 (659.65 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Phase 9.5 — Compile Model

**Goal:** Compile the model using the Adam optimizer, categorical crossentropy loss, and accuracy metrics to configure it for training.

In [22]:
# Cell 19 - Import Adam Optimizer
from tensorflow.keras.optimizers import Adam

In [23]:
# Cell 20 - Compile Model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [24]:
# Cell 21 - Check Compilation Success
print("✅ Model Compiled Successfully!")

✅ Model Compiled Successfully!


## Phase 9.6 — Model Training

**Goal:** Configure callbacks (ModelCheckpoint, EarlyStopping, ReduceLROnPlateau) and train the model.

In [27]:
# Cell 22 - Import Callbacks for Training Optimization
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

In [28]:
# Cell 23 - Configure Model Checkpoint Callback
checkpoint = ModelCheckpoint(
    filepath="../models/best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

In [29]:
# Cell 24 - Configure Early Stopping Callback
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [30]:
# Cell 25 - Configure Learning Rate Reduction Callback
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [31]:
# Cell 26 - Train the Model
EPOCHS = 10

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=validation_dataset,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

c:\Users\DELL\Documents\5th semester\PROJECTS\AI-Powered-Multi-Crop-Disease-Detection-Platform-for-Smart-Agriculture\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/10
 389/1188 ━━━━━━━━━━━━━━━━━━━━ 16:47 1s/step - accuracy: 0.2054 - loss: 3.1802

: 